# Lecture 9 Companion Notebook: Euclidean Equivariance in Physical Systems

This notebook builds **executable intuition** for a central Lecture 9 idea: Euclidean symmetry is not an arbitrary modeling preference, but a property of the space itself. We use short 2D experiments to see how rotations constrain field transformations and kernel forms, and why admissible operators must be built from geometric quantities that survive the symmetry action.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

np.random.seed(0)

plt.rcParams.update({
    "figure.figsize": (6, 5),
    "image.cmap": "viridis",
    "axes.grid": False,
    "font.size": 11,
})


def make_grid(n=121, lim=2.5):
    xs = np.linspace(-lim, lim, n)
    ys = np.linspace(-lim, lim, n)
    X, Y = np.meshgrid(xs, ys)
    return X, Y


def rot_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def rotate_scalar_field(field, theta_deg):
    # s'(u) = s(R^{-1}u) implemented with interpolation on a regular grid.
    return ndimage.rotate(field, angle=theta_deg, reshape=False, order=1, mode="nearest")


def rotate_vector_field(vx, vy, theta_deg):
    # v'(u) = R v(R^{-1}u): first rotate where we sample, then rotate vector values.
    vx_s = ndimage.rotate(vx, angle=theta_deg, reshape=False, order=1, mode="nearest")
    vy_s = ndimage.rotate(vy, angle=theta_deg, reshape=False, order=1, mode="nearest")
    R = rot_matrix(np.deg2rad(theta_deg))
    vx_r = R[0, 0] * vx_s + R[0, 1] * vy_s
    vy_r = R[1, 0] * vx_s + R[1, 1] * vy_s
    return vx_r, vy_r


def plot_scalar(ax, arr, title, extent):
    im = ax.imshow(arr, origin="lower", extent=extent)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.colorbar(im, ax=ax, fraction=0.045, pad=0.04)


def plot_vector(ax, X, Y, vx, vy, title, step=6, color="black"):
    ax.quiver(X[::step, ::step], Y[::step, ::step], vx[::step, ::step], vy[::step, ::step], color=color)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")


def side_by_side_scalar(arrays, titles, extent):
    fig, axs = plt.subplots(1, len(arrays), figsize=(5 * len(arrays), 4))
    if len(arrays) == 1:
        axs = [axs]
    for ax, arr, t in zip(axs, arrays, titles):
        plot_scalar(ax, arr, t, extent)
    plt.tight_layout()


def local_offsets(X, Y, u0, radius):
    z1 = u0[0] - X
    z2 = u0[1] - Y
    r = np.sqrt(z1**2 + z2**2)
    mask = r <= radius
    return z1, z2, r, mask


def phi(r, sigma=0.6):
    return np.exp(-(r**2) / (2 * sigma**2))


## 1. Scalars vs vectors under rotation

A common confusion is to treat scalar and vector fields identically under coordinate changes. The distinction is simple:

- A **scalar** is just a number attached to each point. Under rotation, values do not rotate; sampling locations do.
- A **vector** has direction in space. Under rotation, both sampling locations **and** vector values rotate.

In [ ]:
# Grid and example scalar field
X, Y = make_grid(n=121, lim=2.5)
extent = [X.min(), X.max(), Y.min(), Y.max()]

a = 1.1
b = 0.55
scalar = np.exp(-((X - 0.7) ** 2 / a**2 + (Y + 0.2) ** 2 / b**2))
scalar += 0.25 * np.exp(-((X + 1.0) ** 2 + (Y - 0.9) ** 2) / 0.45**2)

theta_deg = 40
scalar_rot = rotate_scalar_field(scalar, theta_deg)

side_by_side_scalar(
    [scalar, scalar_rot],
    ["Original scalar field $s(u)$", f"Rotated by reindexing $s'(u)=s(R^{{-1}}u)$ ({theta_deg}°)"],
    extent,
)


The color/intensity values are not themselves rotated objects; they are sampled at rotated coordinates. This is the trivial representation behavior for scalar channels.

In [ ]:
# Example vector field: swirl + weak radial component
r2 = X**2 + Y**2 + 1e-6
vx = -Y * np.exp(-0.25 * r2) + 0.15 * X
vy =  X * np.exp(-0.25 * r2) + 0.15 * Y

# WRONG: rotate sample locations but not vector values
vx_wrong = ndimage.rotate(vx, angle=theta_deg, reshape=False, order=1, mode="nearest")
vy_wrong = ndimage.rotate(vy, angle=theta_deg, reshape=False, order=1, mode="nearest")

# CORRECT: v'(u)=R v(R^{-1}u)
vx_corr, vy_corr = rotate_vector_field(vx, vy, theta_deg)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
plot_vector(axs[0], X, Y, vx, vy, "Original vector field")
plot_vector(axs[1], X, Y, vx_wrong, vy_wrong, "WRONG: coordinates rotated only", color="tab:red")
plot_vector(axs[2], X, Y, vx_corr, vy_corr, "CORRECT: coordinates + vectors rotated", color="tab:green")
plt.tight_layout()


The wrong panel violates geometric meaning: arrows keep their old orientation relative to the world axes. The correct panel rotates orientation too. This is exactly why vectors use the standard (nontrivial) representation under rotation.

## 2. Geometry from pairs of points: $z = u_0 - v$

Equivariant kernels are built from relative displacement, not absolute position. Fix a target point $u_0$ and inspect all neighboring points $v$. The intrinsic geometry of each pair is:

- distance $\|z\|$
- direction $z / \|z\|$ (when $z\neq 0$)

In [ ]:
u0 = np.array([0.4, -0.2])
z1, z2, r, mask = local_offsets(X, Y, u0=u0, radius=1.4)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

# Distance map
plot_scalar(axs[0], np.where(mask, r, np.nan), "Distance $||z||$ around target $u_0$", extent)
axs[0].scatter([u0[0]], [u0[1]], c="white", edgecolors="black", s=70, label="$u_0$")
axs[0].legend(loc="upper right")

# Direction field z = u0 - v
zz1 = np.where(mask, z1, np.nan)
zz2 = np.where(mask, z2, np.nan)
axs[1].quiver(X[::6, ::6], Y[::6, ::6], zz1[::6, ::6], zz2[::6, ::6], color="tab:blue")
axs[1].scatter([u0[0]], [u0[1]], c="red", s=60)
axs[1].set_title("Direction vectors $z=u_0-v$")
axs[1].set_xlabel("x")
axs[1].set_ylabel("y")
axs[1].set_aspect("equal")
plt.tight_layout()


## 3. Four canonical kernel types (local view at one target point)

In each case we separate:

1. **Geometry**: displacement $z=u_0-v$
2. **Data**: field value at the neighbor, $x(v)$
3. **Kernel rule**: how geometry and data combine to give the output type

In [ ]:
# Shared local neighborhood and synthetic fields
u0 = np.array([0.0, 0.0])
z1, z2, r, mask = local_offsets(X, Y, u0=u0, radius=1.2)
w = phi(r, sigma=0.55) * mask

# Scalar input field (for A,B)
scalar_in = np.exp(-((X - 0.55) ** 2 + (Y + 0.15) ** 2) / 0.4**2) - 0.6 * np.exp(-((X + 0.6) ** 2 + (Y - 0.2) ** 2) / 0.6**2)

# Vector input field (for C,D)
vec_in_x = 0.8 * X - 0.9 * Y
vec_in_y = 0.5 * X + 0.3 * Y

# A) scalar -> scalar: k(z)=phi(||z||)
contrib_A = w * scalar_in
out_A = contrib_A.sum()

# B) scalar -> vector: k(z)=phi(||z||) z
contrib_Bx = w * z1 * scalar_in
contrib_By = w * z2 * scalar_in
out_B = np.array([contrib_Bx.sum(), contrib_By.sum()])

# C) vector -> scalar: k(z)(x)=phi(||z||)<z,x>
dot_zx = z1 * vec_in_x + z2 * vec_in_y
contrib_C = w * dot_zx
out_C = contrib_C.sum()

# D) vector -> vector: k(z)=phi1 I + phi2 zz^T/||z||^2
phi1 = phi(r, sigma=0.75) * mask
phi2 = 0.7 * phi(r, sigma=0.45) * mask
safe_r2 = np.where(r > 1e-8, r**2, 1.0)

# Isotropic channel
iso_x = phi1 * vec_in_x
iso_y = phi1 * vec_in_y
out_D_iso = np.array([iso_x.sum(), iso_y.sum()])

# Directional channel
proj = (z1 * vec_in_x + z2 * vec_in_y) / safe_r2
dir_x = phi2 * proj * z1
dir_y = phi2 * proj * z2
out_D_dir = np.array([dir_x.sum(), dir_y.sum()])
out_D = out_D_iso + out_D_dir

print("Local outputs at u0:")
print(f"A) scalar->scalar: {out_A:.4f}")
print(f"B) scalar->vector: [{out_B[0]:.4f}, {out_B[1]:.4f}]")
print(f"C) vector->scalar: {out_C:.4f}")
print(f"D) vector->vector: [{out_D[0]:.4f}, {out_D[1]:.4f}]")


### Uniform side-by-side visualization of the four cases

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# A: scalar contributions
imA = axs[0, 0].imshow(np.where(mask, contrib_A, np.nan), origin="lower", extent=extent)
axs[0, 0].set_title(f"A) Scalar→Scalar contributions
$k(z)=\phi(||z||)$, output={out_A:.3f}")
axs[0, 0].scatter([u0[0]], [u0[1]], c="white", edgecolors="black", s=60)
plt.colorbar(imA, ax=axs[0, 0], fraction=0.045, pad=0.04)

# B: vector contributions from scalar input
axs[0, 1].quiver(X[::6, ::6], Y[::6, ::6], contrib_Bx[::6, ::6], contrib_By[::6, ::6], color="tab:purple")
axs[0, 1].scatter([u0[0]], [u0[1]], c="red", s=50)
axs[0, 1].set_title(f"B) Scalar→Vector contributions
$k(z)=\phi(||z||)z$, output={out_B}")
axs[0, 1].set_aspect("equal")

# C: scalar contributions from vector input via inner product
imC = axs[1, 0].imshow(np.where(mask, contrib_C, np.nan), origin="lower", extent=extent)
axs[1, 0].scatter([u0[0]], [u0[1]], c="white", edgecolors="black", s=60)
axs[1, 0].set_title(f"C) Vector→Scalar contributions
$k(z)(x)=\phi(||z||)\langle z,x\rangle$, output={out_C:.3f}")
plt.colorbar(imC, ax=axs[1, 0], fraction=0.045, pad=0.04)

# D: isotropic + directional + combined
axs[1, 1].quiver(X[::6, ::6], Y[::6, ::6], iso_x[::6, ::6], iso_y[::6, ::6], color="tab:blue", alpha=0.7, label="isotropic")
axs[1, 1].quiver(X[::6, ::6], Y[::6, ::6], dir_x[::6, ::6], dir_y[::6, ::6], color="tab:orange", alpha=0.7, label="directional")
axs[1, 1].quiver([u0[0]], [u0[1]], [out_D[0]], [out_D[1]], color="black", scale=2.5, label="combined output")
axs[1, 1].scatter([u0[0]], [u0[1]], c="red", s=50)
axs[1, 1].set_title("D) Vector→Vector channels
$\phi_1 I + \phi_2 zz^T/||z||^2$")
axs[1, 1].legend(loc="upper right", fontsize=9)
axs[1, 1].set_aspect("equal")

for ax in axs.ravel():
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()


**Interpretive summary**

- **A (scalar→scalar):** only distance weighting is needed.
- **B (scalar→vector):** geometry supplies direction via $z$.
- **C (vector→scalar):** use the projection $\langle z, x(v)\rangle$ to contract a vector to a scalar equivariantly.
- **D (vector→vector):** combine an isotropic channel ($I$) and an aligned channel ($zz^T$).

## 4. Counterexamples: what symmetry forbids

Admissible kernels are constrained. Here are two intentionally invalid constructions and simple rotation tests.

In [ ]:
theta = np.deg2rad(35)
R = rot_matrix(theta)

# Counterexample 1: scalar->vector with fixed preferred direction e1
# Invalid rule: F_bad(s) = (sum_w s(v)) * e1

def bad_scalar_to_vector(s_vals, w_vals):
    return np.array([np.sum(w_vals * s_vals), 0.0])

# Counterexample 2: vector->scalar using absolute x-coordinate
# Invalid rule: G_bad(v) = sum_w [X(v) * v_x(v)]

def bad_vector_to_scalar(vx_vals, w_vals, X_vals):
    return np.sum(w_vals * X_vals * vx_vals)

# Prepare local values around u0=0 for clean comparison
u0 = np.array([0.0, 0.0])
z1, z2, r, mask = local_offsets(X, Y, u0=u0, radius=1.2)
w = phi(r, sigma=0.5) * mask

s = scalar_in
vx0, vy0 = vec_in_x, vec_in_y

# Rotate inputs correctly
s_rot = rotate_scalar_field(s, np.rad2deg(theta))
vx_rot, vy_rot = rotate_vector_field(vx0, vy0, np.rad2deg(theta))

bad_out_1 = bad_scalar_to_vector(s, w)
bad_out_1_rot_input = bad_scalar_to_vector(s_rot, w)

bad_out_2 = bad_vector_to_scalar(vx0, w, X)
bad_out_2_rot_input = bad_vector_to_scalar(vx_rot, w, X)

print("Counterexample 1 (scalar->vector, fixed e1):")
print("Rotate output should equal output of rotated input.")
print("R @ F_bad(s)                =", R @ bad_out_1)
print("F_bad(rotated s)            =", bad_out_1_rot_input)

print("
Counterexample 2 (vector->scalar using absolute x):")
print("A scalar output should be unchanged under rotation.")
print("G_bad(v)                    =", bad_out_2)
print("G_bad(rotated v)            =", bad_out_2_rot_input)


Both tests fail: the first introduces an external preferred direction, and the second depends on absolute coordinates. Neither can be justified by Euclidean geometry alone.

## 5. Small numerical rotation checks for the four canonical kernels

We perform local consistency checks at one point. For each operator, we compare:

- **apply operator then rotate output** vs.
- **rotate input then apply operator**

In [ ]:
def eval_A_scalar_to_scalar(s_field, X, Y, u0=np.array([0.0, 0.0]), radius=1.2):
    z1, z2, r, mask = local_offsets(X, Y, u0, radius)
    w = phi(r, sigma=0.55) * mask
    return np.sum(w * s_field)


def eval_B_scalar_to_vector(s_field, X, Y, u0=np.array([0.0, 0.0]), radius=1.2):
    z1, z2, r, mask = local_offsets(X, Y, u0, radius)
    w = phi(r, sigma=0.55) * mask
    ox = np.sum(w * z1 * s_field)
    oy = np.sum(w * z2 * s_field)
    return np.array([ox, oy])


def eval_C_vector_to_scalar(vx_field, vy_field, X, Y, u0=np.array([0.0, 0.0]), radius=1.2):
    z1, z2, r, mask = local_offsets(X, Y, u0, radius)
    w = phi(r, sigma=0.55) * mask
    return np.sum(w * (z1 * vx_field + z2 * vy_field))


def eval_D_vector_to_vector(vx_field, vy_field, X, Y, u0=np.array([0.0, 0.0]), radius=1.2):
    z1, z2, r, mask = local_offsets(X, Y, u0, radius)
    p1 = phi(r, sigma=0.75) * mask
    p2 = 0.7 * phi(r, sigma=0.45) * mask
    safe_r2 = np.where(r > 1e-8, r**2, 1.0)

    iso_x = p1 * vx_field
    iso_y = p1 * vy_field

    proj = (z1 * vx_field + z2 * vy_field) / safe_r2
    dir_x = p2 * proj * z1
    dir_y = p2 * proj * z2

    return np.array([np.sum(iso_x + dir_x), np.sum(iso_y + dir_y)])


theta_deg = 30
R = rot_matrix(np.deg2rad(theta_deg))

s0 = scalar_in
v0x, v0y = vec_in_x, vec_in_y

sR = rotate_scalar_field(s0, theta_deg)
vRx, vRy = rotate_vector_field(v0x, v0y, theta_deg)

A_lhs = eval_A_scalar_to_scalar(sR, X, Y)
A_rhs = eval_A_scalar_to_scalar(s0, X, Y)

B_lhs = eval_B_scalar_to_vector(sR, X, Y)
B_rhs = R @ eval_B_scalar_to_vector(s0, X, Y)

C_lhs = eval_C_vector_to_scalar(vRx, vRy, X, Y)
C_rhs = eval_C_vector_to_scalar(v0x, v0y, X, Y)

D_lhs = eval_D_vector_to_vector(vRx, vRy, X, Y)
D_rhs = R @ eval_D_vector_to_vector(v0x, v0y, X, Y)

print("Rotation consistency checks (small interpolation error is expected):")
print(f"A scalar->scalar |lhs-rhs| = {abs(A_lhs - A_rhs):.3e}")
print(f"B scalar->vector ||lhs-rhs|| = {np.linalg.norm(B_lhs - B_rhs):.3e}")
print(f"C vector->scalar |lhs-rhs| = {abs(C_lhs - C_rhs):.3e}")
print(f"D vector->vector ||lhs-rhs|| = {np.linalg.norm(D_lhs - D_rhs):.3e}")


## 6. Optional spectral perspective (brief)

A plane wave has the form $\exp(i\langle \xi, u\rangle)$. Rotating the signal rotates the oscillation direction, so the frequency vector $\xi$ transforms like a spatial vector. The orbit label is its magnitude $\|\xi\|$, mirroring how radial kernels in space are indexed by $\|z\|$.

In [ ]:
X, Y = make_grid(n=200, lim=3.0)
ext = [X.min(), X.max(), Y.min(), Y.max()]

theta_deg = 35
theta = np.deg2rad(theta_deg)
R = rot_matrix(theta)

xi = np.array([3.2, 1.1])
xi_rot = R @ xi

wave = np.cos(xi[0] * X + xi[1] * Y)
wave_rot_dir = np.cos(xi_rot[0] * X + xi_rot[1] * Y)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plot_scalar(axs[0], wave, f"Plane wave with ξ={xi}", ext)
plot_scalar(axs[1], wave_rot_dir, f"Rotated direction Rξ (θ={theta_deg}°)", ext)
plt.tight_layout()

print(f"||xi|| = {np.linalg.norm(xi):.3f},  ||R xi|| = {np.linalg.norm(xi_rot):.3f}")


## 7. Connection back to Lecture 9 language

- **Euclidean space has intrinsic geometry** (distance and direction from relative displacement).
- **Symmetry groups points into orbits** under rotations/translations.
- **Invariants label orbits** (e.g., $\|z\|$ in space, $\|\xi\|$ in frequency).
- **Valid operators** can only use these invariants plus admissible geometric couplings (e.g., $z$, $\langle z,x\rangle$, $zz^T$).
- The four kernel families above are concrete examples of this principle, not arbitrary architectural choices.

## 8. Exercises (prompts only)

1. Replace the scalar input with a ring-shaped or checkerboard-like scalar field, then rerun the scalar→vector case. How does the output direction at $u_0$ change, and why?

2. Swap the valid scalar→vector kernel with a fixed-direction kernel $k(z)=\phi(\|z\|)e_1$. Run a small rotation test and quantify the mismatch.

3. In the vector→vector case, vary the radial profiles $\phi_1$ and $\phi_2$ (e.g., broader isotropic, sharper directional). Interpret how each channel changes the final vector output.